In [37]:
import pandas as pd

path = 'all_beauty/profiles/train/item_profile.parquet'
df_train = pd.read_parquet(path)

In [39]:
path = 'all_beauty/splits/test.parquet'
df_test = pd.read_parquet(path)

In [40]:
df_test


,rating,title,review,asin,parent_asin,user_id,timestamp,userId,itemId
0,4,Very beautiful shade!,I loved this shade. It was opaque in two coats...,B07RTZMBQC,B07RTZMBQC,AE225THXPLSO5QQ3PHQP7SPETM2A,2019-06-27 17:41:42.843,0,10381
1,5,Five Stars,"Love this product.....unfortunately, I can't a...",B00DT4757A,B00DT4757A,AE22XHMBOBJBXUFCTNYLFMD4UKMA,2016-08-13 16:50:37.000,1,1066
2,5,Must have for nail prep,This set of bits is really great for nail prep...,B0BHWJ7BBJ,B0BHWJ7BBJ,AE23NZUELB4BYWLHKWJXAS73PTSQ,2022-11-24 20:07:50.424,2,18700
3,4,Does The Job; Could Be Cheaper,"While this is a decent costume, it doesn't tak...",B09XF5772H,B09XF4WD3V,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,2022-06-10 01:10:10.648,3,18302
4,5,Polish safe to use while fungus is growing out,It's never a good idea to wear polish if you h...,B072J3L5KV,B0BNWSR278,AE24J5HKPC7GM56SA2CW2HGF3YXQ,2021-06-29 23:25:07.910,4,18760
...,...,...,...,...,...,...,...,...,...
8196,5,Much better than ‘regular’ clippers,Using a straight clipper helps to prevent hang...,B007Y54ABE,B007Y54ABE,AHZYZ2CXUVWNGQTETZVAP3NW5MOA,2022-11-13 23:20:21.731,8196,715
8197,4,Great for brushes,This clear case for makeup brushes works extac...,B0BNJ2KVTP,B0BNJ2KVTP,AHZYZL5IDS67DNZWC3KSKFTR6NXA,2023-03-13 14:14:51.307,8197,18755
8198,5,Loves it,My daughter loved it and it looked great on her,B07CW1NN96,B07CW1NN96,AHZZ3SRJZNF3NRKIAI32UF345O6Q,2019-03-22 16:05:59.022,8198,8044
8199,4,Four Stars,is ok,B01J6I6MJY,B01J6I6MJY,AHZZ6SGWPYHMV7EKQDBVUTHHFZUA,2017-02-03 06:07:21.000,8199,4858


In [23]:
import re

# O parser original usava `:\s*(.+)$`; como `\s` casa com "\n", um campo vazio
# ("Dislikes:  \nContext: ...") capturava a linha seguinte. Aqui o espaço
# após os dois-pontos fica restrito à mesma linha ([ \t]) e campo vazio vira "none".
USER_KEYS = ["Likes", "Dislikes", "Context", "Summary"]
ITEM_KEYS = ["Strengths", "Weaknesses", "Suits", "Summary"]

def match_field(text, k):
    return re.search(rf"^\s*\**{k}\**[ \t]*:[ \t]*(.*)$", text, flags=re.I | re.M)

def parse_fields(text, keys):
    out = {}
    for k in keys:
        m = match_field(text, k)
        value = m.group(1).strip() if m else ""
        out[k.lower()] = value if value else "none"
    out["parse_ok"] = all(match_field(text, k) for k in keys)
    return pd.Series(out)

id_col, keys = ("userId", USER_KEYS) if "userId" in df.columns else ("itemId", ITEM_KEYS)
cols = [k.lower() for k in keys]
fixed = df["raw_output"].apply(parse_fields, keys=keys)
changed = (fixed[cols] != df[cols]).any(axis=1)
print(f"linhas corrigidas: {changed.sum()} de {len(df)}")

df[cols + ["parse_ok"]] = fixed[cols + ["parse_ok"]]
df[df[id_col] == ("23" if id_col == "userId" else "1160")].T

linhas corrigidas: 2105 de 8197


,23
userId,23
likes,colors
dislikes,none
context,none
summary,The user appreciates vibrant colors in beauty ...
raw_output,Likes: colors \nDislikes: \nContext: \nSumm...
parse_ok,True


In [25]:
df.to_parquet(path, index=False)

In [26]:
df

,userId,likes,dislikes,context,summary,raw_output,parse_ok
0,0,"unique and special packaging, opaque formula i...",color not suitable for the user,nail polish collector,The user appreciates nail polishes with unique...,"Likes: unique and special packaging, opaque fo...",True
1,1,"product effectiveness, reordering habit",none,none,The user appreciates products that are effecti...,"Likes: product effectiveness, reordering habit...",True
2,2,"magnetic eyeliner, eyelash customization, YouT...","difficulty in application, irritation from las...",user has trouble applying false eyelashes with...,The user appreciates products like magnetic ey...,"Likes: magnetic eyeliner, eyelash customizatio...",True
3,3,"bath bombs with long-lasting scent, hair wraps...","strong perfume scent in bath products, itchy h...","enjoys spa-like experiences at home, has thick...",This user enjoys beauty products that enhance ...,"Likes: bath bombs with long-lasting scent, hai...",True
4,4,"sulfate-free shampoo, brand consistency with k...","poor value for money, unpleasant smell, lack o...","keratin-treated hair, looking for complementar...",The user prefers sulfate-free shampoos that co...,"Likes: sulfate-free shampoo, brand consistency...",True
...,...,...,...,...,...,...,...
8192,8196,"clear oil, no odor","product not as shown in photo, expired or old ...","intends to use in eyes, concerned about produc...","The user values clear, odorless oils and is co...","Likes: clear oil, no odor \nDislikes: product...",True
8193,8197,"shower caps that keep most hair dry, sleeping ...",hair extensions that are shiny and tangle easily,"naturally curly hair, straightened hair, needs...",This user prefers beauty products that are fun...,"Likes: shower caps that keep most hair dry, sl...",True
8194,8198,"product availability, product effectiveness, e...",none,need to purchase items that are no longer avai...,The user values products that are effective an...,"Likes: product availability, product effective...",True
8195,8199,none,none,none,"The user provided only a single, uninformative...",Likes: \nDislikes: \nContext: \nSummary: The u...,True


In [27]:
df = pd.read_parquet('all_beauty/splits/test.parquet')

In [28]:
df

,rating,title,review,asin,parent_asin,user_id,timestamp,userId,itemId
0,4,Very beautiful shade!,I loved this shade. It was opaque in two coats...,B07RTZMBQC,B07RTZMBQC,AE225THXPLSO5QQ3PHQP7SPETM2A,2019-06-27 17:41:42.843,0,10381
1,5,Five Stars,"Love this product.....unfortunately, I can't a...",B00DT4757A,B00DT4757A,AE22XHMBOBJBXUFCTNYLFMD4UKMA,2016-08-13 16:50:37.000,1,1066
2,5,Must have for nail prep,This set of bits is really great for nail prep...,B0BHWJ7BBJ,B0BHWJ7BBJ,AE23NZUELB4BYWLHKWJXAS73PTSQ,2022-11-24 20:07:50.424,2,18700
3,4,Does The Job; Could Be Cheaper,"While this is a decent costume, it doesn't tak...",B09XF5772H,B09XF4WD3V,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,2022-06-10 01:10:10.648,3,18302
4,5,Polish safe to use while fungus is growing out,It's never a good idea to wear polish if you h...,B072J3L5KV,B0BNWSR278,AE24J5HKPC7GM56SA2CW2HGF3YXQ,2021-06-29 23:25:07.910,4,18760
...,...,...,...,...,...,...,...,...,...
8196,5,Much better than ‘regular’ clippers,Using a straight clipper helps to prevent hang...,B007Y54ABE,B007Y54ABE,AHZYZ2CXUVWNGQTETZVAP3NW5MOA,2022-11-13 23:20:21.731,8196,715
8197,4,Great for brushes,This clear case for makeup brushes works extac...,B0BNJ2KVTP,B0BNJ2KVTP,AHZYZL5IDS67DNZWC3KSKFTR6NXA,2023-03-13 14:14:51.307,8197,18755
8198,5,Loves it,My daughter loved it and it looked great on her,B07CW1NN96,B07CW1NN96,AHZZ3SRJZNF3NRKIAI32UF345O6Q,2019-03-22 16:05:59.022,8198,8044
8199,4,Four Stars,is ok,B01J6I6MJY,B01J6I6MJY,AHZZ6SGWPYHMV7EKQDBVUTHHFZUA,2017-02-03 06:07:21.000,8199,4858
